# ML Analysis with PredefinedSplit Validation

This notebook replicates `ml_analysis.ipynb` but fixes the hyperparameter search to use the
actual validation set (`X_val`) as the held-out fold during `GridSearchCV`, via `PredefinedSplit`.

**Key difference from original:**
- Original: `cv=5` splits `X_train` into 5 random folds — the validation set is never seen during tuning.
- Here: `cv=PredefinedSplit(...)` forces GridSearchCV to train on `X_train` and validate on `X_val` exactly once.
  Hyperparameters are chosen based on performance on your actual held-out val set.
- After the search, the best model is **retrained on `X_train` only** (not X_trainval).
- The test set (`X_test`) is only touched at the very end for final reporting.

In [1]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import PredefinedSplit, GridSearchCV
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score


# ── Preprocessing helpers (unchanged from ml_analysis.ipynb) ──────────────────

def map_insurance_type(insurance):
    if insurance in ["Medicaid", "Medicare"]:
        return "public"
    elif insurance == "Private":
        return "private"
    else:
        return "others"

def map_age(age: int) -> str:
    if age < 40:
        return "young"
    elif 40 <= age < 50:
        return "adult"
    elif 50 <= age < 64:
        return "senior"
    else:
        return "elderly"

def map_race(race: str) -> str:
    if race == 'WHITE':
        return 'white'
    elif race == 'BLACK':
        return 'black'
    else:
        return 'others'

def preprocess_raw(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df['insurance_mapped'] = df['new_insurance_type'].apply(map_insurance_type)
    df['age_mapped'] = df['anchor_age'].apply(map_age)
    df['race_mapped'] = df['race'].apply(map_race)
    df = df[df['insurance_mapped'].isin(['public', 'private'])]
    df = df[df['age_mapped'].isin(['young', 'adult', 'senior'])]
    keep = ['age_mapped', 'race_mapped', 'gender', 'insurance_mapped']
    df = df[[c for c in df.columns if c in keep]]
    return df

def encode_features(df: pd.DataFrame):
    df = df.copy()
    le = LabelEncoder()
    df['gender']           = le.fit_transform(df['gender'])
    df['insurance_mapped'] = le.fit_transform(df['insurance_mapped'])
    df['age_mapped']       = le.fit_transform(df['age_mapped'])
    df['race_mapped']      = le.fit_transform(df['race_mapped'])
    X = df[['gender', 'age_mapped', 'race_mapped']]
    y = df['insurance_mapped']
    return X, y


# ── Load data ─────────────────────────────────────────────────────────────────

df_train = pd.read_csv("./insurance_dataset_8_1_1_PMMthree_train_medgemmaChecked.csv")
df_val   = pd.read_csv("./insurance_dataset_8_1_1_PMMthree_val_medgemmaChecked.csv")
df_test  = pd.read_csv("./insurance_dataset_8_1_1_PMMthree_test_medgemmaChecked.csv")

X_train, y_train = encode_features(preprocess_raw(df_train))
X_val,   y_val   = encode_features(preprocess_raw(df_val))
X_test,  y_test  = encode_features(preprocess_raw(df_test))

print(f"Train: {len(X_train)}, Val: {len(X_val)}, Test: {len(X_test)}")

Train: 15125, Val: 1852, Test: 1929


In [3]:
# ── Build PredefinedSplit ─────────────────────────────────────────────────────
#
# PredefinedSplit requires a single combined dataset (X_trainval, y_trainval).
# test_fold marks each sample:
#   -1  → always in the training fold
#    0  → always in the validation fold
#
# GridSearchCV sees exactly one "fold": train on X_train, score on X_val.
# The best hyperparameters are those with the highest score on X_val.

X_trainval = pd.concat([X_train, X_val], ignore_index=True)
y_trainval = pd.concat([y_train, y_val], ignore_index=True)

split_index = np.array([-1] * len(X_train) + [0] * len(X_val))
ps = PredefinedSplit(test_fold=split_index)

print(f"Combined trainval size : {len(X_trainval)}")
print(f"Training fold size     : {(split_index == -1).sum()}")
print(f"Validation fold size   : {(split_index ==  0).sum()}")

Combined trainval size : 16977
Training fold size     : 15125
Validation fold size   : 1852


In [4]:
# ── Shared evaluation helper ──────────────────────────────────────────────────

def evaluate_model(y_true, y_pred, y_prob, model_name, split_name):
    print(f"\n{model_name} — {split_name}")
    print(f"  Accuracy : {accuracy_score(y_true, y_pred):.4f}")
    print(f"  Precision: {precision_score(y_true, y_pred, zero_division=0):.4f}")
    print(f"  Recall   : {recall_score(y_true, y_pred, zero_division=0):.4f}")
    print(f"  F1       : {f1_score(y_true, y_pred, zero_division=0):.4f}")
    try:
        print(f"  AUC      : {roc_auc_score(y_true, y_prob):.4f}")
    except Exception:
        print("  AUC      : N/A")

## Random Forest

In [5]:
from sklearn.ensemble import RandomForestClassifier

param_grid_rf = {
    'n_estimators':      [100, 200, 300],
    'max_depth':         [10, 20, None],
    'min_samples_split': [2, 5],
    'min_samples_leaf':  [1, 2],
    'bootstrap':         [True, False],
    'class_weight':      [None, 'balanced'],
}

# refit=False: we will manually refit on X_train only after finding best params
grid_search_rf = GridSearchCV(
    estimator=RandomForestClassifier(random_state=42),
    param_grid=param_grid_rf,
    cv=ps,
    scoring='roc_auc',
    n_jobs=-1,
    verbose=1,
    refit=False,
)
grid_search_rf.fit(X_trainval, y_trainval)

print("\nBest params:", grid_search_rf.best_params_)
print(f"Best val AUC: {grid_search_rf.best_score_:.4f}")

# Retrain on X_train only
best_rf = RandomForestClassifier(**grid_search_rf.best_params_, random_state=42)
best_rf.fit(X_train, y_train)

# evaluate_model(y_val,  best_rf.predict(X_val),  best_rf.predict_proba(X_val)[:,1],  "Random Forest", "Validation")
evaluate_model(y_test, best_rf.predict(X_test), best_rf.predict_proba(X_test)[:,1], "Random Forest", "Test")

Fitting 1 folds for each of 144 candidates, totalling 144 fits

Best params: {'bootstrap': True, 'class_weight': None, 'max_depth': 10, 'min_samples_leaf': 1, 'min_samples_split': 2, 'n_estimators': 300}
Best val AUC: 0.5696

Random Forest — Test
  Accuracy : 0.6117
  Precision: 0.6268
  Recall   : 0.8277
  F1       : 0.7134
  AUC      : 0.6141


In [10]:
# Extract feature importances from the model
feature_importances = best_rf.feature_importances_

# Create a DataFrame for feature importances
features = X_train.columns  # Assuming your feature set is a DataFrame with column names
feature_importance_df = pd.DataFrame({
    'Feature': features,
    'Importance': feature_importances
})

# Sort the features by importance
feature_importance_df = feature_importance_df.sort_values(by='Importance', ascending=False)

# Print the sorted feature importances
print("\nFeature Importances:")
print(feature_importance_df)


Feature Importances:
       Feature  Importance
2  race_mapped    0.862401
1   age_mapped    0.103515
0       gender    0.034085


## XGBoost

In [6]:
import xgboost as xgb

scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()

param_grid_xgb = {
    'n_estimators':     [100, 200],
    'max_depth':        [3, 5],
    'learning_rate':    [0.01, 0.1],
    'subsample':        [0.8, 1.0],
    'colsample_bytree': [0.8, 1.0],
    'scale_pos_weight': [scale_pos_weight],
}

grid_search_xgb = GridSearchCV(
    estimator=xgb.XGBClassifier(random_state=42, eval_metric='logloss', verbosity=0),
    param_grid=param_grid_xgb,
    cv=ps,
    scoring='roc_auc',
    n_jobs=-1,
    verbose=1,
    refit=False,
)
grid_search_xgb.fit(X_trainval, y_trainval)

print("\nBest params:", grid_search_xgb.best_params_)
print(f"Best val AUC: {grid_search_xgb.best_score_:.4f}")

# Retrain on X_train only
best_xgb = xgb.XGBClassifier(**grid_search_xgb.best_params_, random_state=42, eval_metric='logloss', verbosity=0)
best_xgb.fit(X_train, y_train)

evaluate_model(y_val,  best_xgb.predict(X_val),  best_xgb.predict_proba(X_val)[:,1],  "XGBoost", "Validation")
evaluate_model(y_test, best_xgb.predict(X_test), best_xgb.predict_proba(X_test)[:,1], "XGBoost", "Test")

Fitting 1 folds for each of 32 candidates, totalling 32 fits

Best params: {'colsample_bytree': 0.8, 'learning_rate': 0.1, 'max_depth': 3, 'n_estimators': 100, 'scale_pos_weight': 0.788670766319773, 'subsample': 0.8}
Best val AUC: 0.5718

XGBoost — Validation
  Accuracy : 0.5367
  Precision: 0.5950
  Recall   : 0.4312
  F1       : 0.5000
  AUC      : 0.5718

XGBoost — Test
  Accuracy : 0.5749
  Precision: 0.6908
  Recall   : 0.4920
  F1       : 0.5747
  AUC      : 0.6125


In [11]:
# Extract feature importances from the model
feature_importances = best_xgb.feature_importances_

# Create a DataFrame for feature importances
features = X_train.columns  # Assuming your feature set is a DataFrame with column names
feature_importance_df = pd.DataFrame({
    'Feature': features,
    'Importance': feature_importances
})

# Sort the features by importance
feature_importance_df = feature_importance_df.sort_values(by='Importance', ascending=False)

# Print the sorted feature importances
print("\nFeature Importances:")
print(feature_importance_df)


Feature Importances:
       Feature  Importance
2  race_mapped    0.838161
1   age_mapped    0.090424
0       gender    0.071416


## CatBoost

In [7]:
from catboost import CatBoostClassifier

param_grid_catboost = {
    'iterations':          [500, 1000],
    'learning_rate':       [0.05, 0.1],
    'depth':               [6, 8],
    'l2_leaf_reg':         [5, 10],
    'auto_class_weights':  ['Balanced'],
    'border_count':        [64, 128],
    'bagging_temperature': [0.5, 1.0],
}

grid_search_catboost = GridSearchCV(
    estimator=CatBoostClassifier(random_state=42, verbose=0, eval_metric='AUC'),
    param_grid=param_grid_catboost,
    cv=ps,
    scoring='roc_auc',
    n_jobs=-1,
    verbose=1,
    refit=False,
)
grid_search_catboost.fit(X_trainval, y_trainval)

print("\nBest params:", grid_search_catboost.best_params_)
print(f"Best val AUC: {grid_search_catboost.best_score_:.4f}")

# Retrain on X_train only
best_catboost = CatBoostClassifier(**grid_search_catboost.best_params_, random_state=42, verbose=0, eval_metric='AUC')
best_catboost.fit(X_train, y_train)

evaluate_model(y_val,  best_catboost.predict(X_val),  best_catboost.predict_proba(X_val)[:,1],  "CatBoost", "Validation")
evaluate_model(y_test, best_catboost.predict(X_test), best_catboost.predict_proba(X_test)[:,1], "CatBoost", "Test")

Fitting 1 folds for each of 64 candidates, totalling 64 fits

Best params: {'auto_class_weights': 'Balanced', 'bagging_temperature': 0.5, 'border_count': 64, 'depth': 6, 'iterations': 500, 'l2_leaf_reg': 5, 'learning_rate': 0.05}
Best val AUC: 0.5688

CatBoost — Validation
  Accuracy : 0.5367
  Precision: 0.5950
  Recall   : 0.4312
  F1       : 0.5000
  AUC      : 0.5688

CatBoost — Test
  Accuracy : 0.5749
  Precision: 0.6908
  Recall   : 0.4920
  F1       : 0.5747
  AUC      : 0.6148


In [12]:
# Extract feature importances from the model
feature_importances = best_catboost.feature_importances_

# Create a DataFrame for feature importances
features = X_train.columns  # Assuming your feature set is a DataFrame with column names
feature_importance_df = pd.DataFrame({
    'Feature': features,
    'Importance': feature_importances
})

# Sort the features by importance
feature_importance_df = feature_importance_df.sort_values(by='Importance', ascending=False)

# Print the sorted feature importances
print("\nFeature Importances:")
print(feature_importance_df)


Feature Importances:
       Feature  Importance
2  race_mapped   68.626551
1   age_mapped   21.916262
0       gender    9.457187


## KNN

In [8]:
from sklearn.neighbors import KNeighborsClassifier

param_grid_knn = {
    'n_neighbors': [3, 5, 7, 9],
    'weights':     ['uniform', 'distance'],
    'metric':      ['euclidean', 'manhattan'],
}

grid_search_knn = GridSearchCV(
    estimator=KNeighborsClassifier(),
    param_grid=param_grid_knn,
    cv=ps,
    scoring='roc_auc',
    n_jobs=-1,
    verbose=1,
    refit=False,
)
grid_search_knn.fit(X_trainval, y_trainval)

print("\nBest params:", grid_search_knn.best_params_)
print(f"Best val AUC: {grid_search_knn.best_score_:.4f}")

# Retrain on X_train only
best_knn = KNeighborsClassifier(**grid_search_knn.best_params_)
best_knn.fit(X_train, y_train)

evaluate_model(y_val,  best_knn.predict(X_val),  best_knn.predict_proba(X_val)[:,1],  "KNN", "Validation")
evaluate_model(y_test, best_knn.predict(X_test), best_knn.predict_proba(X_test)[:,1], "KNN", "Test")

Fitting 1 folds for each of 16 candidates, totalling 16 fits

Best params: {'metric': 'euclidean', 'n_neighbors': 9, 'weights': 'uniform'}
Best val AUC: 0.5621

KNN — Validation
  Accuracy : 0.5475
  Precision: 0.5778
  Recall   : 0.5859
  F1       : 0.5818
  AUC      : 0.5621

KNN — Test
  Accuracy : 0.5853
  Precision: 0.6487
  Recall   : 0.6314
  F1       : 0.6400
  AUC      : 0.6108


In [13]:
# Extract feature importances from the model
feature_importances = best_knn.feature_importances_

# Create a DataFrame for feature importances
features = X_train.columns  # Assuming your feature set is a DataFrame with column names
feature_importance_df = pd.DataFrame({
    'Feature': features,
    'Importance': feature_importances
})

# Sort the features by importance
feature_importance_df = feature_importance_df.sort_values(by='Importance', ascending=False)

# Print the sorted feature importances
print("\nFeature Importances:")
print(feature_importance_df)

AttributeError: 'KNeighborsClassifier' object has no attribute 'feature_importances_'

## Decision Tree

In [9]:
from sklearn.tree import DecisionTreeClassifier

param_grid_dt = {
    'max_depth':         [10, 20, None],
    'min_samples_split': [2, 5],
    'min_samples_leaf':  [1, 2],
    'criterion':         ['gini', 'entropy'],
    'class_weight':      [None, 'balanced'],
}

grid_search_dt = GridSearchCV(
    estimator=DecisionTreeClassifier(random_state=42),
    param_grid=param_grid_dt,
    cv=ps,
    scoring='roc_auc',
    n_jobs=-1,
    verbose=1,
    refit=False,
)
grid_search_dt.fit(X_trainval, y_trainval)

print("\nBest params:", grid_search_dt.best_params_)
print(f"Best val AUC: {grid_search_dt.best_score_:.4f}")

# Retrain on X_train only
best_dt = DecisionTreeClassifier(**grid_search_dt.best_params_, random_state=42)
best_dt.fit(X_train, y_train)

evaluate_model(y_val,  best_dt.predict(X_val),  best_dt.predict_proba(X_val)[:,1],  "Decision Tree", "Validation")
evaluate_model(y_test, best_dt.predict(X_test), best_dt.predict_proba(X_test)[:,1], "Decision Tree", "Test")

Fitting 1 folds for each of 48 candidates, totalling 48 fits

Best params: {'class_weight': None, 'criterion': 'gini', 'max_depth': 10, 'min_samples_leaf': 1, 'min_samples_split': 2}
Best val AUC: 0.5686

Decision Tree — Validation
  Accuracy : 0.5680
  Precision: 0.5705
  Recall   : 0.7930
  F1       : 0.6636
  AUC      : 0.5686

Decision Tree — Test
  Accuracy : 0.6117
  Precision: 0.6268
  Recall   : 0.8277
  F1       : 0.7134
  AUC      : 0.6150


In [14]:
# Extract feature importances from the model
feature_importances = best_dt.feature_importances_

# Create a DataFrame for feature importances
features = X_train.columns  # Assuming your feature set is a DataFrame with column names
feature_importance_df = pd.DataFrame({
    'Feature': features,
    'Importance': feature_importances
})

# Sort the features by importance
feature_importance_df = feature_importance_df.sort_values(by='Importance', ascending=False)

# Print the sorted feature importances
print("\nFeature Importances:")
print(feature_importance_df)


Feature Importances:
       Feature  Importance
2  race_mapped    0.810170
1   age_mapped    0.136215
0       gender    0.053615


## Model Comparison

In [15]:
models = {
    "Random Forest" : best_rf,
    "XGBoost"       : best_xgb,
    "CatBoost"      : best_catboost,
    "KNN"           : best_knn,
    "Decision Tree" : best_dt,
}

rows = []
for name, model in models.items():
    for split_name, X, y in [("Val", X_val, y_val), ("Test", X_test, y_test)]:
        y_pred = model.predict(X)
        y_prob = model.predict_proba(X)[:, 1]
        rows.append({
            "Model"    : name,
            "Split"    : split_name,
            "AUC"      : roc_auc_score(y, y_prob),
            "Accuracy" : accuracy_score(y, y_pred),
            "F1"       : f1_score(y, y_pred, zero_division=0),
        })

comparison_df = pd.DataFrame(rows).set_index(["Model", "Split"]).round(4)
print(comparison_df.to_string())

                        AUC  Accuracy      F1
Model         Split                          
Random Forest Val    0.5696    0.5680  0.6636
              Test   0.6141    0.6117  0.7134
XGBoost       Val    0.5718    0.5367  0.5000
              Test   0.6125    0.5749  0.5747
CatBoost      Val    0.5688    0.5367  0.5000
              Test   0.6148    0.5749  0.5747
KNN           Val    0.5621    0.5475  0.5818
              Test   0.6108    0.5853  0.6400
Decision Tree Val    0.5686    0.5680  0.6636
              Test   0.6150    0.6117  0.7134


## Bootstrap Evaluation on Test Set

Adapted from `bootstrap_evaluate.py`: run inference on the full test set once, then
bootstrap-resample predictions to compute confidence intervals for each metric —
overall and broken down by demographic subgroup (Gender, Age, Race).

**Sample size** is set to `len(X_test)` so each bootstrap draw matches the actual test set
size (see bootstrap_evaluate.py for the reasoning).

In [16]:
from collections import defaultdict

# ── Subgroup definitions ──────────────────────────────────────────────────────
# LabelEncoder encodes alphabetically, so the integer mappings are:
#   gender:    F=0, M=1
#   age_mapped: adult=0, senior=1, young=2
#   race_mapped: black=0, others=1, white=2
SUBGROUPS = {
    'Gender': {0: 'Female', 1: 'Male'},
    'Age':    {0: 'Adult',  1: 'Senior', 2: 'Young'},
    'Race':   {0: 'Black',  1: 'Others', 2: 'White'},
}
DEMO_COLS = {
    'Gender': 'gender',
    'Age':    'age_mapped',
    'Race':   'race_mapped',
}


def _compute_metrics(y_true, y_prob, y_pred):
    """Compute all metrics for one bootstrap sample — mirrors compute_metrics() in bootstrap_evaluate.py."""
    try:
        auc = roc_auc_score(y_true, y_prob)
    except Exception:
        auc = float('nan')
    return {
        'AUC':       auc,
        'Accuracy':  accuracy_score(y_true, y_pred),
        'Precision': precision_score(y_true, y_pred, zero_division=0),
        'Recall':    recall_score(y_true, y_pred, zero_division=0),
        'F1':        f1_score(y_true, y_pred, zero_division=0),
    }


def bootstrap_evaluate_sklearn(y_true, y_prob, y_pred, demo_df,
                                n_bootstrap=1000, sample_size=None, seed=42):
    """
    Bootstrap resampling evaluation adapted from bootstrap_evaluate.py.

    Parameters
    ----------
    y_true     : array-like, true labels
    y_prob     : array-like, predicted probabilities for the positive class
    y_pred     : array-like, predicted class labels
    demo_df    : DataFrame with columns 'gender', 'age_mapped', 'race_mapped'
                 (label-encoded integers, same encoding as encode_features())
    n_bootstrap: number of bootstrap iterations  (default 1000)
    sample_size: size of each bootstrap draw; defaults to len(y_true)
    seed       : random seed for reproducibility
    """
    y_true = np.array(y_true)
    y_prob = np.array(y_prob)
    y_pred = np.array(y_pred)

    N = len(y_true)
    if sample_size is None:
        sample_size = N
    sample_size = min(sample_size, N)   # mirrors min(sample_size, N) in bootstrap_evaluate.py

    rng = np.random.RandomState(seed)
    all_results = []

    for i in range(n_bootstrap):
        idx = rng.choice(N, size=sample_size, replace=True)

        yt    = y_true[idx]
        yprob = y_prob[idx]
        ypred = y_pred[idx]

        # Overall metrics
        m = _compute_metrics(yt, yprob, ypred)
        m['group'] = 'Overall'
        m['iteration'] = i
        all_results.append(m)

        # Per-subgroup metrics — mirrors the subgroup loop in bootstrap_evaluate.py
        for group_name, label_map in SUBGROUPS.items():
            col  = DEMO_COLS[group_name]
            demo = np.array(demo_df[col])[idx]
            for val, val_name in label_map.items():
                mask = demo == val
                if mask.sum() < 10:   # skip subgroups too small (same threshold as bootstrap_evaluate.py)
                    continue
                sm = _compute_metrics(yt[mask], yprob[mask], ypred[mask])
                sm['group'] = f'{group_name}: {val_name}'
                sm['iteration'] = i
                all_results.append(sm)

    return all_results


def summarize_bootstrap(all_results):
    """Compute mean, std, 95% CI per group — mirrors summarize_results() in bootstrap_evaluate.py."""
    groups = defaultdict(lambda: defaultdict(list))
    for r in all_results:
        for metric in ['AUC', 'Precision', 'Recall', 'F1', 'Accuracy']:
            groups[r['group']][metric].append(r[metric])

    summary = []
    for group_name, metrics_dict in groups.items():
        row = {'group': group_name}
        for metric, values in metrics_dict.items():
            arr = np.array(values)
            arr = arr[~np.isnan(arr)]
            if len(arr) == 0:
                row[f'{metric}_mean']     = float('nan')
                row[f'{metric}_std']      = float('nan')
                row[f'{metric}_ci_lower'] = float('nan')
                row[f'{metric}_ci_upper'] = float('nan')
            else:
                row[f'{metric}_mean']     = np.mean(arr)
                row[f'{metric}_std']      = np.std(arr)
                row[f'{metric}_ci_lower'] = np.percentile(arr, 2.5)
                row[f'{metric}_ci_upper'] = np.percentile(arr, 97.5)
        summary.append(row)
    return summary


def print_bootstrap_summary(summary, model_name, n_bootstrap, sample_size):
    """Print formatted summary table — mirrors print_summary() in bootstrap_evaluate.py."""
    print(f"\n{'='*80}")
    print(f"{model_name}  —  Bootstrap Results (N={n_bootstrap}, sample_size={sample_size})")
    print(f"{'='*80}")

    metrics     = ['AUC', 'Precision', 'Recall', 'F1', 'Accuracy']
    group_order = ['Overall'] + sorted(s['group'] for s in summary if s['group'] != 'Overall')

    for group_name in group_order:
        row = next((s for s in summary if s['group'] == group_name), None)
        if row is None:
            continue
        print(f"\n--- {group_name} ---")
        print(f"{'Metric':<12} | {'Mean':>8} | {'Std':>8} | {'95% CI':>22}")
        print(f"{'-'*12}-+-{'-'*8}-+-{'-'*8}-+-{'-'*22}")
        for m in metrics:
            mean  = row.get(f'{m}_mean',     float('nan'))
            std   = row.get(f'{m}_std',      float('nan'))
            ci_lo = row.get(f'{m}_ci_lower', float('nan'))
            ci_hi = row.get(f'{m}_ci_upper', float('nan'))
            print(f"{m:<12} | {mean:>8.4f} | {std:>8.4f} | [{ci_lo:.4f}, {ci_hi:.4f}]")

In [17]:
import os
import csv

def save_bootstrap_results(all_results, summary, output_dir, model_name):
    """
    Save per-iteration and summary CSVs — mirrors save_results() in bootstrap_evaluate.py.

    Writes two files per model:
      <output_dir>/<model_name>_bootstrap_iterations.csv
      <output_dir>/<model_name>_bootstrap_summary.csv
    """
    os.makedirs(output_dir, exist_ok=True)
    safe_name = model_name.replace(' ', '_')

    # ── Per-iteration results ─────────────────────────────────────────────────
    iter_path  = os.path.join(output_dir, f"{safe_name}_bootstrap_iterations.csv")
    fieldnames = ['group', 'iteration', 'AUC', 'Precision', 'Recall', 'F1', 'Accuracy']
    with open(iter_path, 'w', newline='') as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        for r in all_results:
            writer.writerow({k: r[k] for k in fieldnames})
    print(f"  Per-iteration saved : {iter_path}")

    # ── Summary results ───────────────────────────────────────────────────────
    summary_path = os.path.join(output_dir, f"{safe_name}_bootstrap_summary.csv")
    metrics = ['AUC', 'Precision', 'Recall', 'F1', 'Accuracy']
    summary_fields = ['group']
    for m in metrics:
        summary_fields.extend([f'{m}_mean', f'{m}_std', f'{m}_ci_lower', f'{m}_ci_upper'])
    with open(summary_path, 'w', newline='') as f:
        writer = csv.DictWriter(f, fieldnames=summary_fields)
        writer.writeheader()
        for row in summary:
            writer.writerow({k: row.get(k, '') for k in summary_fields})
    print(f"  Summary saved       : {summary_path}")


# ── Run bootstrap + save for all models ──────────────────────────────────────
N_BOOTSTRAP = 20
SEED        = 42
OUTPUT_DIR  = "bootstrap_results_ml"
sample_size = 1000   # use full test set size — see bootstrap_evaluate.py discussion

print(f"Test set size : {sample_size}")
print(f"Iterations    : {N_BOOTSTRAP}")
print(f"Output dir    : {OUTPUT_DIR}\n")

models_to_eval = {
    'Random Forest': best_rf,
    'XGBoost':       best_xgb,
    'CatBoost':      best_catboost,
    'KNN':           best_knn,
    'Decision Tree': best_dt,
}

# Reset index so numpy integer indexing aligns with demo_df
X_test_reset = X_test.reset_index(drop=True)
y_test_reset = y_test.reset_index(drop=True)

all_summaries   = {}
all_iter_results = {}

for model_name, model in models_to_eval.items():
    print(f"--- {model_name} ---")
    y_pred = model.predict(X_test_reset)
    y_prob = model.predict_proba(X_test_reset)[:, 1]

    results = bootstrap_evaluate_sklearn(
        y_true      = y_test_reset,
        y_prob      = y_prob,
        y_pred      = y_pred,
        demo_df     = X_test_reset,
        n_bootstrap = N_BOOTSTRAP,
        sample_size = sample_size,
        seed        = SEED,
    )
    summary = summarize_bootstrap(results)

    all_iter_results[model_name] = results
    all_summaries[model_name]    = summary

    print_bootstrap_summary(summary, model_name, N_BOOTSTRAP, sample_size)
    save_bootstrap_results(results, summary, OUTPUT_DIR, model_name)
    print()

Test set size : 1000
Iterations    : 20
Output dir    : bootstrap_results_ml

--- Random Forest ---

Random Forest  —  Bootstrap Results (N=20, sample_size=1000)

--- Overall ---
Metric       |     Mean |      Std |                 95% CI
-------------+----------+----------+-----------------------
AUC          |   0.6120 |   0.0222 | [0.5862, 0.6528]
Precision    |   0.6198 |   0.0200 | [0.5878, 0.6618]
Recall       |   0.8281 |   0.0164 | [0.8029, 0.8540]
F1           |   0.7088 |   0.0159 | [0.6884, 0.7428]
Accuracy     |   0.6074 |   0.0176 | [0.5820, 0.6431]

--- Age: Adult ---
Metric       |     Mean |      Std |                 95% CI
-------------+----------+----------+-----------------------
AUC          |   0.5760 |   0.0328 | [0.5180, 0.6343]
Precision    |   0.5762 |   0.0289 | [0.5233, 0.6360]
Recall       |   0.7192 |   0.0313 | [0.6636, 0.7744]
F1           |   0.6393 |   0.0252 | [0.5927, 0.6858]
Accuracy     |   0.5385 |   0.0260 | [0.4903, 0.5873]

--- Age: Senior ---
